# Day 044 Project: Inventory Management App

## What You're Building

A data-backed inventory manager using SQLAlchemy ORM with SQLite. Add items, search by category, update prices, delete items — all through Python objects, with no raw SQL.

**Deliverable:** All five CRUD functions wired up, 5+ items managed, and `_run_project_checks()` passes.

## Project Requirements

1. Call `setup_engine()` and open a `Session`
2. Add at least 5 items across at least 2 categories
3. Use `get_items(category=...)` to filter by at least one category
4. Use `update_price` to change at least one price
5. Use `delete_item` to remove at least one item
6. Store all added items in a list called `inventory`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Item(Base):
    __tablename__ = 'items'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    name:     Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    price:    Mapped[float] = mapped_column()
    quantity: Mapped[int]   = mapped_column(default=0)

    def __repr__(self):
        return f'Item(id={self.id}, name={self.name!r}, price={self.price})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def add_item(session, name, category, price, quantity=0):
    item = Item(name=name, category=category, price=price, quantity=quantity)
    session.add(item)
    session.commit()
    session.refresh(item)
    return item


def get_items(session, category=None):
    stmt = select(Item)
    if category is not None:
        stmt = stmt.where(Item.category == category)
    return list(session.execute(stmt).scalars().all())


def update_price(session, item_id, new_price):
    item = session.get(Item, item_id)
    if item is None:
        return None
    item.price = new_price
    session.commit()
    session.refresh(item)
    return item


def delete_item(session, item_id):
    item = session.get(Item, item_id)
    if item is None:
        return False
    session.delete(item)
    session.commit()
    return True


engine  = setup_engine()
session = Session(engine)
print('Inventory DB ready.')

## Step 1 — Stock the Inventory

In [ ]:
inventory = []

# TODO: add at least 5 items in at least 2 categories
# Example:
# inventory.append(add_item(session, 'Laptop',    'Electronics', 999.99, 5))
# inventory.append(add_item(session, 'Headphones','Electronics', 149.99, 12))
# inventory.append(add_item(session, 'Desk Chair','Furniture',   349.00, 3))
# inventory.append(add_item(session, 'Bookcase',  'Furniture',   199.00, 8))
# inventory.append(add_item(session, 'Pen Set',   'Stationery',   12.99, 50))
print(f'Stocked {len(inventory)} items')

## Step 2 — Browse by Category

In [ ]:
# TODO: filter by one of your categories
# category_items = get_items(session, category='Electronics')
# print(f'{len(category_items)} Electronics items:')
# for item in category_items:
#     print(f'  {item}')

## Step 3 — Update a Price

In [ ]:
# TODO: update the price of one item
# updated = update_price(session, inventory[0].id, 849.99)
# print(f'Updated: {updated}')

## Step 4 — Remove an Item

In [ ]:
# TODO: delete one item
# removed = delete_item(session, inventory[-1].id)
# print(f'Deleted: {removed}')
# print(f'Remaining: {len(get_items(session))} items')

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: inventory list has >= 5 items
    try:
        assert 'inventory' in globals(), 'inventory not defined'
        assert len(inventory) >= 5, \
            f'expected >= 5 items in inventory, got {len(inventory)}'
        assert all(isinstance(i, Item) for i in inventory)
        passed += 1; print(f'\u2705 Check 1: {len(inventory)} items stocked')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: get_items returns all persisted items (at least 4 — may have deleted some)
    try:
        all_items = get_items(session)
        assert len(all_items) >= 4, \
            f'expected >= 4 items in DB, got {len(all_items)}'
        passed += 1; print(f'\u2705 Check 2: {len(all_items)} items in DB')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: category filtering works
    try:
        categories = list({i.category for i in get_items(session)})
        assert len(categories) >= 2, \
            f'expected >= 2 categories, got {categories}'
        sample_cat = categories[0]
        filtered = get_items(session, category=sample_cat)
        assert all(i.category == sample_cat for i in filtered)
        passed += 1; print(f'\u2705 Check 3: category filter works ({sample_cat}: {len(filtered)} items)')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: update_price works
    try:
        first = get_items(session)[0]
        new_p = round(first.price * 0.9, 2)
        u = update_price(session, first.id, new_p)
        assert u is not None
        assert abs(u.price - new_p) < 0.01
        passed += 1; print(f'\u2705 Check 4: update_price works (id={first.id} → {new_p})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: delete_item works
    try:
        temp = add_item(session, '_tmp_delete_check', 'Test', 0.01, 1)
        assert delete_item(session, temp.id) is True
        assert session.get(Item, temp.id) is None
        passed += 1; print('\u2705 Check 5: delete_item removes item from DB')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `search_items(session, keyword)` function using `Item.name.ilike(f'%{keyword}%')` (SQLAlchemy case-insensitive LIKE)
- Switch from `sqlite:///:memory:` to `sqlite:///inventory.db` and verify that data persists between sessions
- Add a second model (`Category`) and a ForeignKey relationship, then use `relationship()` to navigate between models
- On Day 45 you will build an ETL pipeline that writes results into a SQLAlchemy-backed database